In [24]:
import gradio as gr
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Khởi tạo mô hình mặc định
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Biến lưu mô hình đã train và dữ liệu sản phẩm
trained_model = None
product_data = []

# Hàm tạo dữ liệu giá cửa hàng giả lập
def generate_data():
    data = [
        ("Giá sản phẩm A là 100,000 VND", 0),
        ("Giá sản phẩm B là 200,000 VND", 1),
        ("Giá sản phẩm C là 150,000 VND", 0)
    ]
    return data

# Hàm train mô hình
def train_model():
    global trained_model, product_data
    data = generate_data()
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    inputs = tokenizer([text for text, _ in data], padding=True, truncation=True, return_tensors="pt")
    labels = torch.tensor([label for _, label in data])
    outputs = model(**inputs, labels=labels)
    trained_model = model
    
    # Lưu lại dữ liệu sản phẩm sau khi train
    product_data = {f"Sản phẩm {chr(65+i)}": text.split("là")[1].strip() for i, (text, _) in enumerate(data)}
    
    return "Mô hình đã được train."

# Hàm hỏi mô hình về giá sản phẩm
def ask_model(question):
    if trained_model is None:
        return "Mô hình chưa được train."
    
    for product, price in product_data.items():
        if product.lower() in question.lower():
            return f"Giá {product} là {price}"
    
    return "Mô hình không có dữ liệu về sản phẩm này."

# Tạo giao diện Gradio
with gr.Blocks() as demo:
    gr.Markdown("# Demo Train Model và Hỏi Đáp về Giá Sản Phẩm")
    
    train_button = gr.Button("Train Model")
    question_input = gr.Textbox(label="Câu hỏi về giá sản phẩm")
    ask_button = gr.Button("Hỏi mô hình")  # Nút hỏi mô hình riêng biệt
    answer_output = gr.Textbox(label="Trả lời")
    
    train_button.click(train_model, outputs=[answer_output])
    ask_button.click(ask_model, inputs=[question_input], outputs=[answer_output])  # Kết nối nút "Hỏi mô hình" với hàm ask_model

demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
